<a href="https://colab.research.google.com/github/parinyad123/financial-analyst-agent/blob/main/notebooks/financial_analyst_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Financial Analyst Agent — Colab Dev Notebook

**Physics-informed financial analysis** ที่ผสม quantitative signals (Hurst exponent) กับ LLM reasoning ผ่าน ReAct agent

| Component | Technology |
|---|---|
| LLM (dev) | Groq — `openai/gpt-oss-120b` (ประหยัด Gemini quota) |
| Agent | LangGraph `create_react_agent` (ReAct pattern) |
| Market data | yfinance |
| Observability | LangSmith — project: `financial-analyst-agent` |

**Flow การทำงาน:**
```
User query → ReAct Agent (gpt-oss-120b)
                ├── get_stock_price      → yfinance
                ├── get_stock_financials → yfinance
                └── get_hurst_exponent   → yfinance + numpy (R/S analysis)
                        ↓
             LangSmith (trace ทุก step)
```

> ⚠️ **ก่อนรัน:** ตั้งค่า Colab Secrets (🔑 ไอคอนซ้ายมือ): `LANGCHAIN_API_KEY`, `GROQ_API_KEY` และเปิด Notebook access

**กฎการรัน:** รันจากบนลงล่างเท่านั้น — ถ้า Cell 2 (Verify) ไม่ผ่าน ห้ามรันต่อ

---
## Cell 1 — Install dependencies + Imports

ติดตั้งทุก package ในที่เดียว แล้ว import ทั้งหมด — รวมไว้ cell เดียวเพื่อไม่ให้เกิดปัญหา import order
(บทเรียนจากรอบ debug: langsmith cache ค่า env ตอน import ครั้งแรก ดังนั้นเราเลิกพึ่ง env แล้วใช้ explicit binding แทน → ดู Cell 2)

In [1]:
# ============================================================
# Cell 1: Install + Imports
# ============================================================
!pip install -q -U langsmith langchain-groq langgraph yfinance langchain-core

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from datetime import datetime

import numpy as np
import yfinance as yf

from google.colab import userdata

# LangChain / LangGraph
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

# LangSmith
import langsmith
from langsmith import Client, traceable, tracing_context
from langsmith.run_helpers import get_current_run_tree

print("✅ All imports OK")
print("langsmith version:", langsmith.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.2/481.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.5 MB/s eta 0:00:00
✅ All imports OK
langsmith version: 0.8.14


---
## Cell 2 — LangSmith client (explicit binding)

**ทำไมไม่ใช้ env vars (`LANGCHAIN_TRACING_V2` ฯลฯ)?**
`langsmith.utils.get_env_var` ถูก cache ด้วย `lru_cache` — ถ้า import ก่อน set env ค่าจะค้างเป็น "disabled" ตลอด session ใน notebook ที่ cell order ไม่แน่นอน นี่คือระเบิดเวลา

**แนวทางที่ใช้:** bind `api_key` + `project_name` ตรง ๆ เข้า `Client`, `LangChainTracer`, และ `@traceable` ทุกจุด — ไม่พึ่ง env เลย
(ตอน deploy จริงใน Docker/FastAPI ค่อยกลับไปใช้ env ได้ เพราะ env ถูก set ตอน container start ก่อน import เสมอ)

**Assertion ท้าย cell:** ถ้า print ❌ ห้ามรัน cell ถัดไป — เช็ค API key ใน Colab Secrets ก่อน

In [2]:
# ============================================================
# Cell 2: LangSmith client + tracer (explicit binding — ไม่พึ่ง env)
# ============================================================
PROJECT_NAME = "financial-analyst-agent"

ls_client = Client(
    api_key=userdata.get("LANGCHAIN_API_KEY"),
    api_url="https://api.smith.langchain.com",
)

# tracer สำหรับ LangGraph agent — ผูก project + client ตรง ๆ
tracer = LangChainTracer(
    project_name=PROJECT_NAME,
    client=ls_client,
)

# สร้าง project ถ้ายังไม่มี + ใช้เป็น connectivity check ไปในตัว
try:
    project = ls_client.read_project(project_name=PROJECT_NAME)
    print(f"✅ Project exists: {project.name}")
except Exception:
    project = ls_client.create_project(
        PROJECT_NAME,
        description="Financial Analyst Agent with ReAct + Hurst Exponent",
    )
    print(f"✅ Project created: {project.name}")

# 🛑 Gate: ต้องผ่านก่อนรันต่อ
assert project is not None, "❌ เชื่อม LangSmith ไม่ได้ — เช็ค LANGCHAIN_API_KEY ใน Colab Secrets"
print("✅ LangSmith ready — ไปต่อได้")

✅ Project exists: financial-analyst-agent
✅ LangSmith ready — ไปต่อได้


---
## Cell 3 — Tools: price, financials, Hurst exponent

**Pattern สำคัญ — `@tool` นอกสุด, `@traceable` ห่อ logic ข้างใน:**

```python
@tool                      # ← LLM เห็น docstring นี้ ใช้ตัดสินใจเลือก tool
def tool_name(x):
    return _logic(x)

@traceable(run_type="tool", client=ls_client)   # ← LangSmith trace ตัวนี้
def _logic(x): ...
```

แยกกันเพราะ: `@tool` ทำหน้าที่ interface กับ LLM (schema + docstring) ส่วน `@traceable` ทำหน้าที่ observability — ถ้อยซ้อน decorator บนฟังก์ชันเดียวกันจะตีกัน

| Tool | ข้อมูลที่คืน | Tags ใน LangSmith |
|---|---|---|
| `get_stock_price` | ราคา, % change, 52W range, P/E, market cap | `market-data`, `yfinance` |
| `get_stock_financials` | revenue, net income, margin, growth, EPS, D/E | `fundamentals`, `yfinance` |
| `get_hurst_exponent` | Hurst exponent (R/S analysis) + market regime | `quant`, `regime-detection` |

**Hurst exponent อ่านยังไง:** H > 0.55 → Trending (momentum ใช้ได้) | H < 0.45 → Mean-Reverting (RSI/Bollinger) | ระหว่างนั้น → Random Walk

In [3]:
# ============================================================
# Cell 3: Tools — get_stock_price / get_stock_financials / get_hurst_exponent
# ============================================================

# ---------- Tool 1: ราคาปัจจุบัน + key metrics ----------
@tool
def get_stock_price(ticker: str) -> str:
    """Fetch current stock price and key metrics."""
    return _fetch_stock_price_logic(ticker)

@traceable(
    name="fetch_stock_price",
    run_type="tool",
    tags=["market-data", "yfinance"],
    client=ls_client,
)
def _fetch_stock_price_logic(ticker: str) -> str:
    try:
        stock = yf.Ticker(ticker.upper())
        hist = stock.history(period="5d")   # 5d เผื่อวันที่ market ปิด
        if hist.empty:
            return f"No data for {ticker}"

        latest = hist['Close'].iloc[-1]
        prev = hist['Close'].iloc[-2] if len(hist) > 1 else latest
        change_pct = ((latest - prev) / prev) * 100
        info = stock.info

        return (
            f"Ticker: {ticker.upper()}\n"
            f"Price: ${latest:.2f} (Change: {change_pct:+.2f}%)\n"
            f"52W Range: ${info.get('fiftyTwoWeekLow','N/A')} – ${info.get('fiftyTwoWeekHigh','N/A')}\n"
            f"P/E (TTM): {info.get('trailingPE','N/A')} | Forward P/E: {info.get('forwardPE','N/A')}\n"
            f"Market Cap: ${info.get('marketCap',0)/1e9:.1f}B"
        )
    except Exception as e:
        return f"Error: {str(e)}"


# ---------- Tool 2: Fundamentals ----------
@tool
def get_stock_financials(ticker: str) -> str:
    """Get fundamental financial metrics for analysis."""
    return _fetch_financials_logic(ticker)

@traceable(
    name="fetch_financials",
    run_type="tool",
    tags=["fundamentals", "yfinance"],
    client=ls_client,
)
def _fetch_financials_logic(ticker: str) -> str:
    try:
        info = yf.Ticker(ticker.upper()).info
        return (
            f"Revenue (TTM): ${info.get('totalRevenue',0)/1e9:.1f}B\n"
            f"Net Income: ${info.get('netIncomeToCommon',0)/1e9:.1f}B\n"
            f"Profit Margin: {info.get('profitMargins',0)*100:.1f}%\n"
            f"Revenue Growth YoY: {info.get('revenueGrowth',0)*100:.1f}%\n"
            f"EPS (TTM): ${info.get('trailingEps','N/A')}\n"
            f"Debt/Equity: {info.get('debtToEquity','N/A')}"
        )
    except Exception as e:
        return f"Error: {str(e)}"


# ---------- Tool 3: Hurst exponent (R/S analysis) ----------
@tool
def get_hurst_exponent(ticker: str) -> str:
    """Calculate Hurst exponent to detect market regime."""
    return _calc_hurst_logic(ticker)

@traceable(
    name="calc_hurst_exponent",
    run_type="tool",
    tags=["quant", "regime-detection"],
    client=ls_client,
)
def _calc_hurst_logic(ticker: str) -> str:
    try:
        # 1) log returns จาก 1Y daily close
        hist = yf.Ticker(ticker.upper()).history(period="1y")["Close"]
        returns = np.log(hist / hist.shift(1)).dropna().values

        # 2) Rescaled Range (R/S) ต่อ lag — แบ่ง series เป็น segments
        lags = range(2, 20)
        rs_values = []
        for lag in lags:
            segments = [returns[i:i+lag] for i in range(0, len(returns)-lag, lag)]
            rs_list = [
                (np.max(np.cumsum(s - np.mean(s))) - np.min(np.cumsum(s - np.mean(s)))) / np.std(s)
                for s in segments if np.std(s) > 0
            ]
            if rs_list:
                rs_values.append(np.mean(rs_list))

        # 3) Hurst = slope ของ log(R/S) vs log(lag)
        hurst = np.polyfit(np.log(list(lags)[:len(rs_values)]), np.log(rs_values), 1)[0]

        # 4) จำแนก regime
        if hurst > 0.55:
            regime = "📈 Trending — momentum strategies work"
        elif hurst < 0.45:
            regime = "↔️ Mean-Reverting — RSI/Bollinger strategies work"
        else:
            regime = "🎲 Random Walk — harder to predict"

        return f"Hurst Exponent ({ticker.upper()}, 1Y): {hurst:.4f}\nRegime: {regime}"
    except Exception as e:
        return f"Error: {str(e)}"


print("✅ Tools ready:", [t.name for t in [get_stock_price, get_stock_financials, get_hurst_exponent]])

✅ Tools ready: ['get_stock_price', 'get_stock_financials', 'get_hurst_exponent']


---
## Cell 4–6 — Tools เพิ่มเติม (TODO ตาม build order)

- **Cell 4:** `analyze_portfolio_risk` (UC-2a) — Volatility, Sharpe, Sortino, VaR/CVaR 95%, Max Drawdown, correlation matrix
- **Cell 5:** `search_market_news` — Gemini 2.0 Flash + Google Search grounding (ใช้ model แยกจาก agent หลัก)
- **Cell 6:** `track_portfolio` (UC-2b) — dev ด้วย `MOCK_PORTFOLIO` dict ก่อน, swap เป็น PostgreSQL ตอน deploy

In [4]:
# ============================================================
# Cell 4: TODO — analyze_portfolio_risk (UC-2a)
# Input: {ticker: weight}, weights ต้องรวม 1.0 ± 0.01
# ============================================================

# Mock portfolio สำหรับทดสอบ UC-2b (Cell 6)
MOCK_PORTFOLIO = {
    "positions": [
        {"ticker": "NVDA", "shares": 10, "avg_cost": 150.0},
        {"ticker": "AMD",  "shares": 20, "avg_cost": 100.0},
        {"ticker": "TSLA", "shares":  5, "avg_cost": 200.0},
    ]
}

---
## Cell 7 — Agent setup (Groq + ReAct)

**ทำไม Groq ไม่ใช่ Gemini ตอน dev:** Gemini free tier ถอด quota ของ `gemini-2.0-flash` ออกแล้ว (`limit: 0` → 429 ตลอด) — dev ใน Colab ใช้ Groq แล้วค่อย swap เป็น Gemini ตอน deploy

**ทำไม `gpt-oss-120b` ไม่ใช่ Llama 3.3 70B:** reasoning model ที่ train มาเพื่อ agentic tasks → tool orchestration เสถียรกว่า + ถูกกว่า (`reasoning_effort="low"` พอสำหรับ tool routing และประหยัด output tokens)

⚠️ Trade-off: ภาษาไทยของ gpt-oss อ่อนกว่า Llama 3.3 (ที่รองรับไทย official) — ถ้า output ไทยเพี้ยน สลับ model ได้ด้วยการเปลี่ยน string เดียว

In [5]:
# ============================================================
# Cell 7: Agent setup — ChatGroq + create_react_agent
# ============================================================

# Dev: Groq | Production: swap เป็น ChatGoogleGenerativeAI(model="gemini-2.0-flash")
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    reasoning_effort="low",   # พอสำหรับ tool orchestration, ประหยัด tokens
    api_key=userdata.get("GROQ_API_KEY"),
)

tools = [get_stock_price, get_stock_financials, get_hurst_exponent]

SYSTEM_PROMPT = """You are a quantitative financial analyst assistant.
Always fetch real-time data before answering.
Provide objective analysis with data. Note that this is not financial advice.
Do not give specific price targets, entry points, or stop-loss levels.
Respond in Thai mixed with English technical terms."""

agent_graph = create_react_agent(model, tools, prompt=SYSTEM_PROMPT)
print("✅ Agent graph ready")

✅ Agent graph ready


---
## Cell 8 — `run_financial_agent` (entry point + tracing wrapper)

หน้าที่ของ wrapper นี้:
1. **`@traceable`** — group ทุก sub-runs (LLM calls + tool calls) ไว้ใน 1 parent trace
2. **`callbacks=[tracer]`** — ส่ง LangGraph internal runs เข้า LangSmith แบบ explicit (ไม่พึ่ง env)
3. **`metadata` + `tags`** — filter ใน LangSmith UI ได้ตาม ticker / analysis_type
4. **`run_id` ใน return** — ดึงจาก `get_current_run_tree()` ภายใน function → ได้ ID ของ trace นี้เป๊ะ ๆ ไม่ต้อง query ย้อนหลัง (และคือ `trace_id` ที่ FastAPI endpoint ต้อง return ตาม spec)

In [6]:
# ============================================================
# Cell 8: run_financial_agent — main entry point
# ============================================================

@traceable(
    name="financial_analyst_agent",
    run_type="chain",
    tags=["agent", "financial-analysis"],
    project_name=PROJECT_NAME,
    client=ls_client,
)
def run_financial_agent(
    query: str,
    tickers: list[str] = None,
    analysis_type: str = "general",
) -> dict:
    """Main entry point — group ทุก sub-runs ไว้ใน 1 parent trace"""
    config = RunnableConfig(
        run_name=f"query_{analysis_type}_{datetime.now().strftime('%H%M%S')}",
        callbacks=[tracer],                       # explicit tracer — ไม่พึ่ง env
        tags=[analysis_type] + (tickers or []),
        metadata={
            "query": query,
            "tickers": tickers or [],
            "analysis_type": analysis_type,
            "timestamp": datetime.now().isoformat(),
        },
    )

    inputs = {"messages": [HumanMessage(content=query)]}
    final_response = ""

    print(f"\n{'='*55}")
    print(f"🔍 Query: {query[:80]}...")
    print(f"{'='*55}")

    # stream_mode="values" → ได้ state เต็มทุก step, print ทุก message (Human/AI/Tool)
    for event in agent_graph.stream(inputs, config=config, stream_mode="values"):
        if "messages" in event:
            last = event["messages"][-1]
            last.pretty_print()
            if hasattr(last, "content") and last.content:
                final_response = last.content

    # ดึง run ID ของ trace นี้จากข้างใน — แม่นกว่า list_runs ย้อนหลัง
    rt = get_current_run_tree()
    return {
        "query": query,
        "response": final_response,
        "tickers": tickers,
        "analysis_type": analysis_type,
        "run_id": str(rt.id) if rt else None,
    }

print("✅ run_financial_agent ready")

✅ run_financial_agent ready


---
## Cell 9 — Test UC-1: วิเคราะห์หุ้นรายตัว

ลำดับการทำงาน:
1. **`tracing_context(enabled=True, client=ls_client)`** — เปิด tracing ให้ `@traceable` ทุกตัวในก้อนนี้ (จำเป็นเพราะเราไม่ได้ set env)
2. **`ls_client.flush()`** — บังคับส่ง pending traces ทันที (ปกติส่งแบบ background batch)
3. **Trace URLs** — private URL (เปิดดูเองใน workspace) + public URL จาก `share_run()` (แปะใน README ให้คนอื่นดูได้โดยไม่ต้อง login)

In [7]:
# ============================================================
# Cell 9: Test UC-1 — single stock analysis
# ============================================================

with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)",
        tickers=["NVDA"],
        analysis_type="full_analysis",
    )

ls_client.flush()   # บังคับส่ง traces ก่อน query หา run

# ---------- Trace URLs ----------
import time
time.sleep(5)       # เผื่อ server-side ingest

run_id = result["run_id"]
run = ls_client.read_run(run_id)
print(f"\n🔒 Private URL: {run.url}")

# Public URL — uncomment ถ้าต้องการ share (เช่นแปะใน README)
# shared_url = ls_client.share_run(run_id)
# print(f"🌐 Public URL: {shared_url}")


🔍 Query: วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)...
================================ Human Message =================================

วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (fc_7a1cfe46-04f5-4e7c-8954-cf67aeedee27)
 Call ID: fc_7a1cfe46-04f5-4e7c-8954-cf67aeedee27
  Args:
    ticker: NVDA
================================= Tool Message =================================
Name: get_stock_price

Ticker: NVDA
Price: $201.69 (Change: +0.63%)
52W Range: $140.85 – $236.54
P/E (TTM): 30.886677 | Forward P/E: 15.847224
Market Cap: $4885.1B
================================== Ai Message ==================================
Tool Calls:
  get_stock_financials (fc_9fb33880-3220-427f-9a39-c603e3a3ec8b)
 Call ID: fc_9fb33880-3220-427f-9a39-c603e3a3ec8b
  Args:
    ticker: NVDA
================================= Tool Me

---
## Cell 10–11 — Test UC-2a / UC-2b (TODO)

รอ tools จาก Cell 4–6 ก่อน:
- **UC-2a:** `run_financial_agent(query="ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%", analysis_type="portfolio_risk")`
- **UC-2b:** track P&L จาก `MOCK_PORTFOLIO`

---
## Utility — ดู runs ย้อนหลังในโปรเจกต์

`list_runs(is_root=True)` เหมาะกับ sanity check / ไล่ดู history หลาย runs — ต่างจาก `result["run_id"]` ที่ได้ run ของ query นั้นเป๊ะ ๆ

In [8]:
# ============================================================
# Utility: list recent root runs
# ============================================================
runs = list(ls_client.list_runs(
    project_name=PROJECT_NAME,
    is_root=True,
    limit=5,
))
for r in runs:
    print(r.start_time, "|", r.name, "|", r.url)

2026-06-11 15:37:47.883906+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb755-186b-71c3-8bf7-4dff1a80008a?trace_id=019eb755-186b-71c3-8bf7-4dff1a80008a&start_time=2026-06-11T15:37:47.883906
2026-06-11 06:57:32.969879+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb578-cb29-73a0-accd-e8bcb6e98e27?trace_id=019eb578-cb29-73a0-accd-e8bcb6e98e27&start_time=2026-06-11T06:57:32.969879
2026-06-11 05:55:38.480082+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb540-1d70-7e53-8c05-26e9591406a1?trace_id=019eb540-1d70-7e53-8c05-26e9591406a1&start_time=2026-06-11T05:55:38.480082
2026-06-11 05:53:26.424345+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff